<center><h1>Fan_Jinning_HW3</h1></center>
<br>
<br>

Name: Jinning Fan
<br>
Github Username: jinningf-cyber
<br>
USC ID: 4608286954

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import io
import zipfile
from pathlib import Path

import requests
import pandas as pd
import numpy as np
from IPython.display import display

Get the AReM Data Set

In [2]:
url = (
    "https://archive.ics.uci.edu/static/public/366/"
    "activity+recognition+system+based+on+"
    "multisensor+data+fusion+arem.zip"
)

response = requests.get(url, timeout=60)
response.raise_for_status()

data_dir = Path("AReM")
data_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    z.extractall(data_dir)

activities = [
    "bending1", "bending2", "cycling", "lying",
    "sitting", "standing", "walking"
]

series_names = [
    "avg_rss12", "var_rss12",
    "avg_rss13", "var_rss13",
    "avg_rss23", "var_rss23"
]

files = sorted(
    [
        file for file in data_dir.rglob("dataset*.csv")
        if file.parent.name in activities
    ],
    key=lambda file: (
        activities.index(file.parent.name),
        int(file.stem.replace("dataset", ""))
    )
)

def read_instance(file):
    # Handle comma-separated and whitespace-separated rows.
    rows = [
        line.replace(",", " ").split()
        for line in file.read_text().splitlines()
        if line.strip() and not line.lstrip().startswith("#")
    ]

    if any(len(row) != 7 for row in rows):
        raise ValueError(f"Unexpected number of columns: {file}")

    data = pd.DataFrame(
        rows,
        columns=["time"] + series_names,
        dtype=float
    )

    if not np.isfinite(data.to_numpy()).all():
        raise ValueError(f"Missing or non-finite values: {file}")

    return data

instances = {}
records = []

for instance_id, file in enumerate(files, start=1):
    data = read_instance(file)
    instances[instance_id] = data[series_names]

    records.append({
        "Instance": instance_id,
        "Activity": file.parent.name,
        "Dataset": int(file.stem.replace("dataset", "")),
        "Rows": len(data)
    })

metadata = pd.DataFrame(records).set_index("Instance")

assert len(instances) == 88

print("Number of instances:", len(instances))
print("Time series per instance:", len(series_names))

display(
    metadata.groupby("Activity").agg(
        Instances=("Dataset", "size"),
        Min_rows=("Rows", "min"),
        Max_rows=("Rows", "max")
    ).reindex(activities)
)

display(instances[1].head())

Number of instances: 88
Time series per instance: 6


,Instances,Min_rows,Max_rows
Activity,,,
bending1,7,480,480
bending2,6,480,480
cycling,15,480,480
lying,15,480,480
sitting,15,479,480
standing,15,480,480
walking,15,480,480


,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,39.25,0.43,22.75,0.43,33.75,1.3
1,39.25,0.43,23.00,0.00,33.00,0.0
2,39.25,0.43,23.25,0.43,33.00,0.0
3,39.50,0.50,23.00,0.71,33.00,0.0
4,39.50,0.50,24.00,0.00,33.00,0.0


### (b) Test and Train Data

In [3]:
# Split by the dataset numbers specified in the assignment.
is_bending = metadata["Activity"].isin(["bending1", "bending2"])

is_test = (
    (is_bending & metadata["Dataset"].isin([1, 2]))
    | (~is_bending & metadata["Dataset"].isin([1, 2, 3]))
)

metadata["Split"] = np.where(is_test, "Test", "Train")

train_ids = metadata.index[metadata["Split"] == "Train"]
test_ids = metadata.index[metadata["Split"] == "Test"]

train_data = {
    instance_id: instances[instance_id]
    for instance_id in train_ids
}

test_data = {
    instance_id: instances[instance_id]
    for instance_id in test_ids
}

split_summary = pd.crosstab(
    metadata["Activity"], metadata["Split"]
).reindex(activities)[["Train", "Test"]]

display(split_summary)

print("Training instances:", len(train_data))
print("Test instances:", len(test_data))

assert len(train_data) == 69
assert len(test_data) == 19

Split,Train,Test
Activity,,
bending1,5,2
bending2,4,2
cycling,12,3
lying,12,3
sitting,12,3
standing,12,3
walking,12,3


Training instances: 69
Test instances: 19


### (c) Feature Extraction

#### i. Research

Common time-domain features include minimum, maximum, mean, median, variance, standard deviation, quartiles, range, and interquartile range. These describe the level and spread of a time series. Skewness and kurtosis describe its distribution shape. Other features include root mean square, signal energy, autocorrelation, and the number of peaks, which capture signal magnitude and temporal patterns.

For this assignment, I use minimum, maximum, mean, median, standard deviation, first quartile, and third quartile.

#### ii. Extraction

In [4]:
# Extract seven features from each of the six time series.
feature_rows = []

for instance_id, data in instances.items():
    row = {"Instance": instance_id}

    for j, column in enumerate(series_names, start=1):
        series = data[column]

        row[f"min_{j}"] = series.min()
        row[f"max_{j}"] = series.max()
        row[f"mean_{j}"] = series.mean()
        row[f"median_{j}"] = series.median()
        row[f"std_{j}"] = series.std(ddof=1)
        row[f"q1_{j}"] = series.quantile(0.25)
        row[f"q3_{j}"] = series.quantile(0.75)

    feature_rows.append(row)

features = pd.DataFrame(feature_rows).set_index("Instance")

X_train = features.loc[train_ids].copy()
X_test = features.loc[test_ids].copy()

y_train = metadata.loc[train_ids, "Activity"].copy()
y_test = metadata.loc[test_ids, "Activity"].copy()

print("Feature matrix shape:", features.shape)
print("Training feature matrix shape:", X_train.shape)
print("Test feature matrix shape:", X_test.shape)
print("Missing feature values:", features.isna().sum().sum())

assert features.shape == (88, 42)
assert features.isna().sum().sum() == 0

with pd.option_context("display.max_columns", None):
    display(features.head().round(3))

Feature matrix shape: (88, 42)
Training feature matrix shape: (69, 42)
Test feature matrix shape: (19, 42)
Missing feature values: 0


,min_1,max_1,mean_1,median_1,std_1,q1_1,q3_1,min_2,max_2,mean_2,median_2,std_2,q1_2,q3_2,min_3,max_3,mean_3,median_3,std_3,q1_3,q3_3,min_4,max_4,mean_4,median_4,std_4,q1_4,q3_4,min_5,max_5,mean_5,median_5,std_5,q1_5,q3_5,min_6,max_6,mean_6,median_6,std_6,q1_6,q3_6
Instance,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.625,40.50,1.477,39.25,42.00,0.0,1.30,0.359,0.43,0.323,0.00,0.50,4.0,29.50,19.041,19.250,4.463,16.00,23.250,0.0,7.23,0.833,0.50,0.966,0.00,1.120,27.25,38.25,34.311,35.0,2.188,33.000,36.00,0.0,1.92,0.571,0.43,0.583,0.0,1.30
2,38.00,45.67,42.813,42.50,1.436,42.00,43.67,0.0,1.22,0.372,0.47,0.289,0.00,0.50,2.0,29.50,20.097,21.000,3.894,19.00,22.250,0.0,5.76,0.876,0.50,1.047,0.00,1.145,27.67,38.50,33.025,33.0,1.995,32.000,34.50,0.0,3.11,0.571,0.43,0.601,0.0,1.30
3,35.00,47.40,43.954,44.33,1.559,43.00,45.00,0.0,1.70,0.426,0.47,0.339,0.00,0.50,6.5,29.75,22.122,23.000,3.031,19.75,24.000,0.0,4.44,0.497,0.43,0.551,0.00,0.830,29.00,38.50,35.588,36.0,2.000,35.362,36.50,0.0,1.79,0.493,0.43,0.514,0.0,0.94
4,33.00,47.75,42.180,43.50,3.671,39.15,45.00,0.0,3.00,0.696,0.50,0.631,0.00,1.12,8.5,30.00,22.184,23.000,3.810,20.50,24.372,0.0,5.15,0.990,0.83,0.954,0.43,1.300,20.00,38.67,33.494,35.0,3.849,30.458,36.33,0.0,2.18,0.614,0.50,0.524,0.0,1.00
5,33.00,45.75,41.678,41.75,2.243,41.33,42.75,0.0,2.83,0.536,0.50,0.405,0.43,0.71,3.0,28.25,19.007,19.125,4.087,16.50,22.062,0.0,6.42,0.842,0.50,0.929,0.43,1.120,23.67,37.50,29.857,30.0,2.411,28.458,31.25,0.0,1.79,0.383,0.43,0.389,0.0,0.50


I extracted seven features from each of the six time series, giving 42 features per instance. The feature matrix has 88 rows and 42 columns. The suffixes 1 through 6 follow the order avg_rss12, var_rss12, avg_rss13, var_rss13, avg_rss23, and var_rss23.

I used sample standard deviation with ddof=1 and kept the features on their original scales. The activity labels and train-test split are stored separately from the feature matrix.

#### iii. Standard Deviation

In [5]:
# Estimate feature standard deviations across all 88 instances.
feature_sd = features.std(ddof=1)

# Construct 90% percentile bootstrap confidence intervals.
rng = np.random.default_rng(42)
n_bootstrap = 10000

values = features.to_numpy()
n_instances, n_features = values.shape

bootstrap_sd = np.empty((n_bootstrap, n_features))

# Resample the complete with replacement.
for b in range(n_bootstrap):
    indices = rng.integers(
        0, n_instances, size=n_instances
    )

    sample = values[indices]
    bootstrap_sd[b] = sample.std(axis=0, ddof=1)

ci_lower, ci_upper = np.percentile(
    bootstrap_sd, [5, 95], axis=0
)

sd_results = pd.DataFrame({
    "Standard deviation": feature_sd,
    "90% CI lower": ci_lower,
    "90% CI upper": ci_upper
})

sd_results.index.name = "Feature"

with pd.option_context("display.max_rows", None):
    display(sd_results.round(4))

,Standard deviation,90% CI lower,90% CI upper
Feature,,,
min_1,9.5700,8.2360,10.7739
max_1,4.3944,3.3399,5.2931
mean_1,5.3357,4.7038,5.8758
median_1,5.4401,4.7840,6.0063
std_1,1.7722,1.5694,1.9495
q1_1,6.1536,5.5738,6.6486
q3_1,5.1389,4.3386,5.8340
min_2,0.0000,0.0000,0.0000
max_2,5.0627,4.6306,5.4011


I estimated the sample standard deviation of each feature across the 88 instances. I then generated 10,000 bootstrap samples by sampling 88 instances with replacement. For each sample, I recalculated the standard deviation of all 42 features.

The 5th and 95th percentiles of the bootstrap estimates give a 90% confidence interval for each feature's standard deviation. These intervals describe uncertainty in the variation across instances, rather than variation within a single time series. I used a fixed random seed so the results are reproducible.

#### iv. Select Features

I would select mean, standard deviation, and maximum. The mean describes the overall signal level, while standard deviation measures how much the signal changes within an instance. The maximum captures high signal values that may help distinguish activities with different movement patterns.

These features summarize different aspects of the time series. Applying them to all six time series gives 18 selected features. This is a judgment-based choice; their predictive usefulness would need to be checked using the training data.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic regression will have training RSS less than or equal to that of the linear regression. The linear model is a special case of the cubic model with the coefficients of X^2 and X^3 set to zero. Adding these terms cannot increase the minimum training RSS, even when the true relationship is linear.

### (b) Linear Test

I would generally expect the linear regression to have lower test RSS when the true relationship is linear. The extra terms in the cubic model are unnecessary and may fit noise in the training data, increasing variance. However, this is an expectation rather than a guarantee for every test set.

### (c) Not Linear Train

The cubic regression will still have training RSS less than or equal to that of the linear regression. This follows from the models being nested and does not depend on the true relationship. The cubic model can always reproduce the linear fit by setting the coefficients of X^2 and X^3 to zero.

### (d) Not Linear Testing

There is not enough information to determine which model will have lower test RSS. If the nonlinear pattern is substantial and the cubic model captures it well, the cubic regression may reduce bias enough to improve prediction. If the relationship is close to linear, the additional variance may outweigh that improvement, so the linear regression may perform better.